### Imports

In [1]:
import os
import sys
import json
import shutil
import imagehash
from tqdm import tqdm
from PIL import Image
from pathlib import Path
from collections import defaultdict
import cv2
import numpy as np
from pycocotools.coco import COCO
import matplotlib.pyplot as plt
from IPython.display import display, Image

sys.path.insert(0, "../../")
from config import DATASETS_PATH, SEED, IMG_SHAPE

### Functions

In [2]:
def merge_image_folders(image_dirs, output_image_dir):
    """
    Copies images from multiple source directories to a single destination directory.

    Args:
        image_dirs (list): A list of paths to the source image folders.
        output_image_dir (str): The path to the folder where all images will be saved.
    """
    print("Starting image copy process...")
    total_images_copied = 0
    os.makedirs(output_image_dir, exist_ok=True)

    for source_dir in image_dirs:
        if not os.path.isdir(source_dir):
            print(f"Warning: Image directory '{source_dir}' does not exist. Skipping.")
            continue
            
        image_files = os.listdir(source_dir)
        print(f"Copying {len(image_files)} images from '{source_dir}'...")
        
        for image_name in image_files:
            source_path = os.path.join(source_dir, image_name)
            destination_path = os.path.join(output_image_dir, image_name)
            shutil.copy2(source_path, destination_path)
            total_images_copied += 1

    print(f"\n✨ Copy finished! A total of {total_images_copied} images have been copied to '{output_image_dir}'.")

In [3]:
def merge_coco_annotations(annotation_files, output_json_path):
    """
    Merges multiple COCO annotation files, re-indexing the IDs to avoid conflicts.

    Args:
        annotation_files (list): A list of paths to the annotation JSON files.
        output_json_path (str): The path where the merged JSON file will be saved.
    """
    print("\nStarting COCO annotation files merge...")
    
    valid_annotation_files = [f for f in annotation_files if os.path.exists(f)]
    if not valid_annotation_files:
        print("Error: No valid annotation files were found.")
        return

    # Load the first file as a base
    with open(valid_annotation_files[0], 'r') as f:
        merged_coco = json.load(f)

    max_image_id = max(img['id'] for img in merged_coco.get('images', [])) if merged_coco.get('images') else 0
    max_annotation_id = max(ann['id'] for ann in merged_coco.get('annotations', [])) if merged_coco.get('annotations') else 0

    print(f"Base file '{valid_annotation_files[0]}' loaded. Max Image ID: {max_image_id}, Max Annotation ID: {max_annotation_id}")

    # Iterate over the rest of the files
    for ann_file in valid_annotation_files[1:]:
        print(f"Processing file: '{ann_file}'...")
        with open(ann_file, 'r') as f:
            data = json.load(f)
        
        image_id_mapping = {}

        # Re-index images
        for image in data.get('images', []):
            old_image_id = image['id']
            max_image_id += 1
            new_image_id = max_image_id
            
            image_id_mapping[old_image_id] = new_image_id
            image['id'] = new_image_id
            merged_coco['images'].append(image)

        # Re-index annotations
        for ann in data.get('annotations', []):
            old_image_id = ann['image_id']
            if old_image_id in image_id_mapping:
                max_annotation_id += 1
                ann['id'] = max_annotation_id
                ann['image_id'] = image_id_mapping[old_image_id]
                merged_coco['annotations'].append(ann)

    # Save the result
    with open(output_json_path, 'w') as f:
        json.dump(merged_coco, f, indent=4)

    print(f"\n✨ Annotation merge finished! File saved at: '{output_json_path}'")

In [4]:
def unify_coco_dataset(base_dir, splits, output_dir):
    """
    Main function that orchestrates the merging of a split COCO dataset.

    Args:
        base_dir (str): The parent folder containing the splits and annotation files.
        splits (list): A list of strings with the split names (e.g., ['train', 'valid']).
        output_dir (str): The destination folder for the merged dataset.
    """
    print(f"--- Starting dataset merge in '{base_dir}' ---")
    
    # 1. Define input and output paths
    image_dirs = [os.path.join(base_dir, split) for split in splits]
    annotation_files = [os.path.join(base_dir, f'annotations_coco_{split}.json') for split in splits]
    
    output_image_dir = os.path.join(output_dir, 'images')
    output_json_path = os.path.join(output_dir, 'annotations_coco.json')
    
    # 2. Call the helper functions
    merge_image_folders(image_dirs, output_image_dir)
    merge_coco_annotations(annotation_files, output_json_path)
    
    print(f"\n✅ Process completed. The merged dataset is located at: '{output_dir}'")

In [5]:
def clean_coco_annotations(json_path, images_dir_path, output_json_path):
    """
    Cleans a COCO JSON file by filtering images and annotations.

    This function removes:
    1. Images that are not present in the specified images directory.
    2. Images that have no corresponding entries in the 'annotations' list.
    3. Any annotations associated with the removed images.

    Args:
        json_path (str): The file path to the original COCO JSON.
        images_dir_path (str): The path to the directory containing the actual image files.
        output_json_path (str): The file path to save the cleaned COCO JSON.
    """
    try:
        # 1. Load the COCO JSON file
        with open(json_path, 'r') as f:
            coco_data = json.load(f)
        print(f"Loaded original JSON from: {json_path}")

        # 2. Get the set of all actual filenames from the image directory
        # Using a set for fast O(1) average-case lookups
        actual_filenames = set(os.listdir(images_dir_path))
        if not actual_filenames:
            print(f"Warning: No image files found in directory: {images_dir_path}")
            return

        print(f"Found {len(actual_filenames)} actual image files in: {images_dir_path}")

        # 3. Get the set of all image IDs that have at least one annotation
        annotated_image_ids = set(ann['image_id'] for ann in coco_data['annotations'])
        print(f"Found {len(annotated_image_ids)} images with annotations.")

        # 4. Filter the 'images' list
        original_image_count = len(coco_data['images'])
        
        filtered_images = [
            img for img in coco_data['images']
            if img['file_name'] in actual_filenames and img['id'] in annotated_image_ids
        ]
        
        new_image_count = len(filtered_images)
        print(f"Filtered 'images' list: Kept {new_image_count} out of {original_image_count}.")

        # 5. Get the set of valid image IDs from the newly filtered list
        valid_image_ids = set(img['id'] for img in filtered_images)

        # 6. Filter the 'annotations' list based on the valid image IDs
        original_annotation_count = len(coco_data['annotations'])
        
        filtered_annotations = [
            ann for ann in coco_data['annotations']
            if ann['image_id'] in valid_image_ids
        ]
        
        new_annotation_count = len(filtered_annotations)
        print(f"Filtered 'annotations' list: Kept {new_annotation_count} out of {original_annotation_count}.")

        # 7. Update the COCO data dictionary
        coco_data['images'] = filtered_images
        coco_data['annotations'] = filtered_annotations

        # 8. Save the cleaned data to the new JSON file
        with open(output_json_path, 'w') as f:
            json.dump(coco_data, f, indent=4)
        
        print(f"\n✅ Successfully saved cleaned COCO JSON to: {output_json_path}")

    except FileNotFoundError:
        print(f"Error: The file or directory was not found. Please check your paths.")
        print(f"  JSON path: {json_path}")
        print(f"  Images path: {images_dir_path}")
    except Exception as e:
        print(f"An error occurred: {e}")


In [6]:
def process_ina_annotation(coco_path, output_path):
    """
    This function will process the COCO annotations from INA experts and give it a standard format 
    where the phases of cells will be in the cateogry_id field instead of in the attributes field.
    This is in order to process the annotations with supervision library.
    """

    # --- Validations ---
    if not os.path.isfile(coco_path):
        print(f"COCO JSON file not found: {coco_path}")
        return

    # --- Configuration ---
    # Adjust these paths to match your project structure
    new_coco_path = os.path.join(output_path, 'annotations_coco.json')

    # --- Preprocessing Steps ---

    # 1. Create the output directory if it doesn't exist
    os.makedirs(output_path, exist_ok=True)

    # 2. Load the original COCO JSON data
    print(f"Loading original annotations from: {coco_path}")
    with open(coco_path, 'r') as f:
        coco_data = json.load(f)

    print("Preprocessing annotations...")

    # 3. Discover unique class names from the 'Fase' attribute and create a mapping
    # Using OrderedDict to maintain a consistent order
    class_names = sorted(list(set(
        ann.get('attributes', {}).get('Fase').replace('f','ph').lower()
        for ann in coco_data['annotations']
        if ann.get('attributes', {}).get('Fase') is not None
    )))

    # Create a mapping from class name to a new category ID (starting from 1)
    class_to_id = {name: i + 1 for i, name in enumerate(class_names)}
    print(f"\nDiscovered and mapped classes: {class_to_id}")

    # 4. Create the new categories list for the output JSON
    new_categories = [
        {"id": cat_id, "name": name, "supercategory": ""}
        for name, cat_id in class_to_id.items()
    ]

    # 5. Create a new list of annotations with updated category_id
    new_annotations = []
    for ann in coco_data['annotations']:
        attributes = ann.get('attributes', {})
        fase = attributes.get('Fase').replace('f','ph').lower()

        # Only include annotations that have a 'Fase' we are interested in
        if fase in class_to_id:
            new_ann = ann.copy()
            new_ann['category_id'] = class_to_id[fase]
            new_annotations.append(new_ann)

    # 6. Assemble the new COCO data structure
    new_coco_data = {
        'licenses': coco_data.get('licenses', []),
        'info': coco_data.get('info', {}),
        'categories': new_categories,
        'images': coco_data.get('images', []),
        'annotations': new_annotations
    }

    # 7. Save the new COCO JSON file
    print(f"\nSaving processed annotations to: {new_coco_path}")
    with open(new_coco_path, 'w') as f:
        json.dump(new_coco_data, f, indent=4)

    print("\nPreprocessing complete!")
    print(f"You can now use '{new_coco_path}' with supervision.")
    return new_coco_path



In [7]:
def rename_and_move_ina_tagged_images(images_dir, images_output_dir, annotations_path):
    """
    Code to rename the tagged images to have the same name as in the coco annotations
    """
    os.makedirs(images_output_dir, exist_ok=True)

    images_paths = [images_dir + '/' + item for item in sorted(os.listdir(images_dir))]

    with open(annotations_path, 'r') as f: #json with the information of the filename of the images
        data = json.load(f)

    for image in tqdm(images_paths):
        image_id = os.path.basename(image).split('.')[0]

        if image_id.isnumeric() == False:
            print('Ignoring non-numeric image:', image)
            continue

        value = data['images'][int(image_id) - 1]['file_name']
        shutil.copy2(image, f'{images_output_dir}/{value}')

In [8]:
def delete_similar_duplicates(directory, threshold=5):
    """
    Finds and directly deletes visually similar images in a directory.
    It keeps the first occurrence and deletes subsequent similar images.

    Args:
        directory (str): The path to the directory containing images.
        threshold (int): The maximum Hamming distance to consider images as duplicates.
                         Lower is stricter. 0 means very similar. Default is 5.
    """
    print(f"Scanning for similar duplicates in '{directory}' with threshold={threshold}...")
    hashes = {}
    files_deleted_count = 0
    
    # 1. Calculate and store the hash for each image
    filepaths = []
    for dirpath, _, filenames in os.walk(directory):
        for filename in sorted(filenames): # Sort to have a deterministic process
            filepaths.append(os.path.join(dirpath, filename))

    for filepath in filepaths:
        try:
            with Image.open(filepath) as img:
                img_hash = imagehash.phash(img)
                hashes[filepath] = img_hash
        except Exception as e:
            print(f"Warning: Could not process file {filepath}: {e}")

    # 2. Compare hashes and delete duplicates
    files_to_keep = set()
    for i in range(len(filepaths)):
        path1 = filepaths[i]
        if path1 in files_to_keep or path1 not in hashes:
            continue # Already processed or failed to hash

        # The first time we see a file, we decide to keep it
        files_to_keep.add(path1)

        for j in range(i + 1, len(filepaths)):
            path2 = filepaths[j]
            if path2 in files_to_keep or path2 not in hashes:
                continue

            distance = hashes[path1] - hashes[path2]
            if distance <= threshold:
                print(f"  - Deleting '{os.path.basename(path2)}' (similar to '{os.path.basename(path1)}', distance={distance})")
                try:
                    os.remove(path2)
                    files_deleted_count += 1
                    # To avoid trying to open it again, mark it as 'processed' by removing its hash
                    del hashes[path2] 
                except OSError as e:
                    print(f"Error deleting file {path2}: {e}")
    
    print(f"\nScan complete. Deleted {files_deleted_count} similar duplicate images.")

In [9]:
def update_coco_annotations(json_path, image_dir):
    """
    Synchronizes a COCO JSON file with the contents of an image directory.
    It removes image and annotation entries from the JSON if the corresponding
    image file does not exist in the directory.

    Args:
        json_path (str): Path to the COCO annotations JSON file.
        image_dir (str): Path to the directory containing the actual image files.
    """
    print(f"\nSynchronizing '{json_path}' with directory '{image_dir}'...")
    
    # 1. Get the set of actual filenames on disk for fast lookup
    try:
        actual_files_on_disk = set(os.listdir(image_dir))
    except FileNotFoundError:
        print(f"Error: Image directory not found at '{image_dir}'")
        return

    # 2. Load the COCO data
    with open(json_path, 'r') as f:
        coco_data = json.load(f)

    # 3. Filter the 'images' list
    original_image_count = len(coco_data['images'])
    images_to_keep = [img for img in coco_data['images'] if img['file_name'] in actual_files_on_disk]
    
    if len(images_to_keep) == original_image_count:
        print("✅ JSON is already in sync. No changes needed.")
        return

    # 4. Get the IDs of the images we are keeping
    kept_image_ids = {img['id'] for img in images_to_keep}

    # 5. Filter the 'annotations' list based on the kept image IDs
    original_annotation_count = len(coco_data['annotations'])
    annotations_to_keep = [ann for ann in coco_data['annotations'] if ann['image_id'] in kept_image_ids]

    # 6. Build the new, cleaned COCO data structure
    cleaned_coco_data = {
        'info': coco_data.get('info', {}),
        'licenses': coco_data.get('licenses', []),
        'images': images_to_keep,
        'annotations': annotations_to_keep,
        'categories': coco_data.get('categories', [])
    }

    # 7. Overwrite the original JSON file
    with open(json_path, 'w') as f:
        json.dump(cleaned_coco_data, f, indent=4)

    print(f"Synchronization complete.")
    print(f"  - Removed {original_image_count - len(images_to_keep)} image entries.")
    print(f"  - Removed {original_annotation_count - len(annotations_to_keep)} annotation entries.")
    print("✅ JSON file updated successfully.")

### Standarize ina dataset

In [10]:
INA_BASE_PATH = os.path.join(DATASETS_PATH, 'full_fov', 'original', 'ina')
IMAGES_DIR =  os.path.join(INA_BASE_PATH, 'tagged_images', 'input')
COCO_ANNOTATIONS_PATH = os.path.join(INA_BASE_PATH, 'tagged_images', 'corte-27-02-2024.json')
CLEANED_COCO_ANNOTATIONS_PATH = os.path.join(INA_BASE_PATH, 'tagged_images', 'corte-27-02-2024_cleaned.json')
OUTPUT_DIR = os.path.join(DATASETS_PATH, 'full_fov', 'processed', 'ina')

rename_and_move_ina_tagged_images(IMAGES_DIR, os.path.join(OUTPUT_DIR, 'images'), COCO_ANNOTATIONS_PATH)
clean_coco_annotations(COCO_ANNOTATIONS_PATH, IMAGES_DIR, CLEANED_COCO_ANNOTATIONS_PATH)
process_ina_annotation(CLEANED_COCO_ANNOTATIONS_PATH, OUTPUT_DIR)

100%|██████████| 58/58 [00:00<00:00, 208100.63it/s]

Ignoring non-numeric image: /home/nicolas/Documentos/UTN/INA/giar_ina_dev/datasets/full_fov/original/ina/tagged_images/input/004_00094.jpg
Ignoring non-numeric image: /home/nicolas/Documentos/UTN/INA/giar_ina_dev/datasets/full_fov/original/ina/tagged_images/input/004_00095.jpg
Ignoring non-numeric image: /home/nicolas/Documentos/UTN/INA/giar_ina_dev/datasets/full_fov/original/ina/tagged_images/input/004_00096.jpg
Ignoring non-numeric image: /home/nicolas/Documentos/UTN/INA/giar_ina_dev/datasets/full_fov/original/ina/tagged_images/input/004_00097.jpg
Ignoring non-numeric image: /home/nicolas/Documentos/UTN/INA/giar_ina_dev/datasets/full_fov/original/ina/tagged_images/input/004_00098.jpg
Ignoring non-numeric image: /home/nicolas/Documentos/UTN/INA/giar_ina_dev/datasets/full_fov/original/ina/tagged_images/input/004_00099.jpg
Ignoring non-numeric image: /home/nicolas/Documentos/UTN/INA/giar_ina_dev/datasets/full_fov/original/ina/tagged_images/input/004_00100.jpg
Ignoring non-numeric image:

'/home/nicolas/Documentos/UTN/INA/giar_ina_dev/datasets/full_fov/processed/ina/annotations_coco.json'

### Unify onion_cell_merged dataset

In [11]:
ONION_CELL_MERGED_BASE_DIR = os.path.join(DATASETS_PATH, 'full_fov', 'original', 'onion_cell_merged')
IMAGES_BASE_DIR = os.path.join(ONION_CELL_MERGED_BASE_DIR, 'images')
SPLITS = ['train', 'valid', 'test']
OUTPUT_DIR = os.path.join(DATASETS_PATH, 'full_fov', 'processed', 'onion_cell_merged')

unify_coco_dataset(base_dir=IMAGES_BASE_DIR, splits=SPLITS, output_dir=OUTPUT_DIR)

--- Starting dataset merge in '/home/nicolas/Documentos/UTN/INA/giar_ina_dev/datasets/full_fov/original/onion_cell_merged/images' ---
Starting image copy process...
Copying 452 images from '/home/nicolas/Documentos/UTN/INA/giar_ina_dev/datasets/full_fov/original/onion_cell_merged/images/train'...
Copying 129 images from '/home/nicolas/Documentos/UTN/INA/giar_ina_dev/datasets/full_fov/original/onion_cell_merged/images/valid'...
Copying 63 images from '/home/nicolas/Documentos/UTN/INA/giar_ina_dev/datasets/full_fov/original/onion_cell_merged/images/test'...

✨ Copy finished! A total of 644 images have been copied to '/home/nicolas/Documentos/UTN/INA/giar_ina_dev/datasets/full_fov/processed/onion_cell_merged/images'.

Starting COCO annotation files merge...
Base file '/home/nicolas/Documentos/UTN/INA/giar_ina_dev/datasets/full_fov/original/onion_cell_merged/images/annotations_coco_train.json' loaded. Max Image ID: 451, Max Annotation ID: 14601
Processing file: '/home/nicolas/Documentos/UT

### Unify roboflow datasets

In [12]:
# Unify each dataset split first
ROBOFLOW_BASE_DIR = os.path.join(DATASETS_PATH, 'full_fov', 'original', 'roboflow_datasets')
DATASETS = ['Mitosis.v1i.coco', 'Mitosis.v15-5.coco', 'Mitosis Counter.v7i.coco', 'mitosis_baseline.v1i.coco']
SPLITS = ['train', 'valid', 'test']
OUTPUT_DIR = os.path.join(DATASETS_PATH, 'full_fov', 'processed', 'roboflow_datasets')

unified_datasets_paths = []
unified_annotations_paths = []

for dataset in DATASETS:
    dataset_path = os.path.join(ROBOFLOW_BASE_DIR, dataset)
    unified_datasets_paths.append(os.path.join(dataset_path, 'unified', 'images'))
    unified_annotations_paths.append(os.path.join(dataset_path, 'unified', 'annotations_coco.json'))
    unify_coco_dataset(base_dir= dataset_path, splits=SPLITS, output_dir=os.path.join(dataset_path, 'unified'))

# Unify all dataset into a single one
UNIFIED_DIR = os.path.join(OUTPUT_DIR)
UNIFIED_IMAGES_DIR = os.path.join(UNIFIED_DIR, 'images')
UNIFIED_ANNOTATIONS = os.path.join(UNIFIED_DIR, 'annotations_coco.json')

merge_image_folders(unified_datasets_paths, UNIFIED_IMAGES_DIR)
merge_coco_annotations(unified_annotations_paths, UNIFIED_ANNOTATIONS)
for path in unified_datasets_paths:
    path_to_remove = os.path.dirname(path)
    shutil.rmtree(path_to_remove)

# Remove similar duplicates
delete_similar_duplicates(UNIFIED_DIR)

--- Starting dataset merge in '/home/nicolas/Documentos/UTN/INA/giar_ina_dev/datasets/full_fov/original/roboflow_datasets/Mitosis.v1i.coco' ---
Starting image copy process...
Copying 10 images from '/home/nicolas/Documentos/UTN/INA/giar_ina_dev/datasets/full_fov/original/roboflow_datasets/Mitosis.v1i.coco/train'...
Copying 1 images from '/home/nicolas/Documentos/UTN/INA/giar_ina_dev/datasets/full_fov/original/roboflow_datasets/Mitosis.v1i.coco/valid'...
Copying 1 images from '/home/nicolas/Documentos/UTN/INA/giar_ina_dev/datasets/full_fov/original/roboflow_datasets/Mitosis.v1i.coco/test'...

✨ Copy finished! A total of 12 images have been copied to '/home/nicolas/Documentos/UTN/INA/giar_ina_dev/datasets/full_fov/original/roboflow_datasets/Mitosis.v1i.coco/unified/images'.

Starting COCO annotation files merge...
Base file '/home/nicolas/Documentos/UTN/INA/giar_ina_dev/datasets/full_fov/original/roboflow_datasets/Mitosis.v1i.coco/annotations_coco_train.json' loaded. Max Image ID: 28, Ma

In [13]:
# Before running this, check visually if there are more images similar images that should be deleted
update_coco_annotations(UNIFIED_ANNOTATIONS, UNIFIED_DIR)


Synchronizing '/home/nicolas/Documentos/UTN/INA/giar_ina_dev/datasets/full_fov/processed/roboflow_datasets/annotations_coco.json' with directory '/home/nicolas/Documentos/UTN/INA/giar_ina_dev/datasets/full_fov/processed/roboflow_datasets'...
Synchronization complete.
  - Removed 1924 image entries.
  - Removed 5523 annotation entries.
✅ JSON file updated successfully.
